In [ ]:
# Install required packages if not already installed

import os
from typing import Dict, List
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

# Initialize the language model
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.1)



## Section 1: Creating Simple Tools

Let's create three basic tools. Each tool will be used by a different specialized agent.

In [2]:
@tool
def get_weather(city: str) -> str:
    """Get weather information for a city."""
    # Simulate weather data
    weather_data = {
        "new york": "Sunny, 72°F",
        "london": "Rainy, 60°F", 
        "tokyo": "Cloudy, 68°F",
        "paris": "Partly cloudy, 65°F",
        "sydney": "Windy, 70°F"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 15.0) -> str:
    """Calculate tip amount and total bill."""
    tip_amount = bill_amount * (tip_percentage / 100)
    total_amount = bill_amount + tip_amount
    
    return f"Bill: ${bill_amount:.2f}, Tip ({tip_percentage}%): ${tip_amount:.2f}, Total: ${total_amount:.2f}"

@tool
def search_recipes(cuisine: str) -> str:
    """Search for recipes based on cuisine type."""
    recipes = {
        "italian": ["Spaghetti Carbonara", "Margherita Pizza", "Risotto"],
        "mexican": ["Tacos", "Quesadillas", "Guacamole"],
        "chinese": ["Fried Rice", "Sweet and Sour Chicken", "Dumplings"],
        "indian": ["Curry", "Biryani", "Samosas"]
    }
    
    cuisine_recipes = recipes.get(cuisine.lower(), ["No recipes found for that cuisine"])
    return f"Popular {cuisine.title()} recipes: {', '.join(cuisine_recipes)}"

# Test the tools individually
print("Testing tools:")
print(f"Weather: {get_weather.invoke({'city': 'New York'})}")
print(f"Tip: {calculate_tip.invoke({'bill_amount': 50.0, 'tip_percentage': 20.0})}")
print(f"Recipes: {search_recipes.invoke({'cuisine': 'italian'})}")

Testing tools:
Weather: Sunny, 72°F
Tip: Bill: $50.00, Tip (20.0%): $10.00, Total: $60.00
Recipes: Popular Italian recipes: Spaghetti Carbonara, Margherita Pizza, Risotto


## Section 2: Creating Specialized Agents

Now let's create three specialized agents. Each agent only knows how to use ONE tool.

In [3]:
# Weather Agent - only handles weather queries
weather_agent = create_react_agent(
    model=llm,
    tools=[get_weather],  # Only has access to weather tool
    prompt=(
        "You are a Weather Agent. You specialize in providing weather information for cities.\n"
        "Use the get_weather tool to look up weather data.\n"
        "Always be helpful and provide the weather information clearly."
    ),
    name="weather_agent"
)

# Tip Calculator Agent - only handles tip calculations
tip_agent = create_react_agent(
    model=llm,
    tools=[calculate_tip],  # Only has access to tip calculation tool
    prompt=(
        "You are a Tip Calculator Agent. You specialize in calculating tips and total bills.\n"
        "Use the calculate_tip tool to help users with bill calculations.\n"
        "Always show the breakdown clearly: bill amount, tip amount, and total."
    ),
    name="tip_agent"
)

# Recipe Agent - only handles recipe searches
recipe_agent = create_react_agent(
    model=llm,
    tools=[search_recipes],  # Only has access to recipe search tool
    prompt=(
        "You are a Recipe Agent. You specialize in finding recipes for different cuisines.\n"
        "Use the search_recipes tool to find popular dishes.\n"
        "Be enthusiastic about food and provide helpful cooking suggestions."
    ),
    name="recipe_agent"
)

print("Three specialized agents created:")
print("Weather Agent - handles weather queries")
print("Tip Agent - handles tip calculations")
print("Recipe Agent - handles recipe searches")

Three specialized agents created:
Weather Agent - handles weather queries
Tip Agent - handles tip calculations
Recipe Agent - handles recipe searches


## Section 3: Testing Individual Agents

Let's test each agent individually to see how they work with their single tool.

In [4]:
def test_agent(agent, message, agent_name):
    """Helper function to test individual agents"""
    print(f"\nTesting {agent_name}")
    print(f"User: {message}")
    print("Agent working...")
    
    response = agent.invoke({"messages": [{"role": "user", "content": message}]})
    final_message = response["messages"][-1].content
    
    print(f"Response: {final_message}")
    print("-" * 50)
    return response

# Test each agent with appropriate queries
test_agent(weather_agent, "What's the weather in London?", "Weather Agent")
test_agent(tip_agent, "Calculate tip for a $85 bill with 18% tip", "Tip Agent")
test_agent(recipe_agent, "Show me some Mexican recipes", "Recipe Agent")


Testing Weather Agent
User: What's the weather in London?
Agent working...
Response: The weather in London is rainy with a temperature of 60°F.
--------------------------------------------------

Testing Tip Agent
User: Calculate tip for a $85 bill with 18% tip
Agent working...
Response: Here's the breakdown for your $85 bill with an 18% tip:

- **Bill Amount:** $85.00
- **Tip (18.0%):** $15.30
- **Total:** $100.30

If you need any further assistance, feel free to ask!
--------------------------------------------------

Testing Recipe Agent
User: Show me some Mexican recipes
Agent working...
Response: Oh, I’m so excited to share some delicious Mexican recipes with you! Here are a few popular dishes that you can try:

1. **Tacos**: These are a classic! You can fill them with anything from seasoned beef, chicken, or fish to beans and veggies. Don’t forget to top them with fresh cilantro, onions, and a squeeze of lime!

2. **Quesadillas**: A quick and easy dish! Just fill a tortilla with

{'messages': [HumanMessage(content='Show me some Mexican recipes', additional_kwargs={}, response_metadata={}, id='f3d0c920-beb3-41af-a2b7-2f4f5380b7f3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_zPIMa3kgf7ogEJwfGubPeyhf', 'function': {'arguments': '{"cuisine":"Mexican"}', 'name': 'search_recipes'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 87, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CFuuSzImOBxRPDpUrXQ4iqQRorwoW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='recipe_agent', id='run--d3257845-a3f3-4b4b-a49b-9ec51b2a403b-0', tool_calls=[{'name': 'search_recipes', 'args':

## Section 4: Creating the Supervisor Agent

Now let's create a supervisor that can coordinate all three agents and pick the right one for each task.

In [6]:
# Create the supervisor that manages all three agents
supervisor = create_supervisor(
    model=llm,
    agents=[weather_agent, tip_agent, recipe_agent],
    prompt=(
        "You are a Supervisor that coordinates three specialized agents:\n\n"
        "1. WEATHER AGENT: Handles weather-related questions (weather, temperature, forecast)\n"
        "2. TIP AGENT: Handles tip calculations and bill-related math\n"
        "3. RECIPE AGENT: Handles cooking and recipe questions\n\n"
        "Your job is to:\n"
        "- Analyze the user's question\n"
        "- Pick the RIGHT agent for the task\n"
        "- Delegate the work to that agent\n"
        "- Return the agent's response to the user\n\n"
        "IMPORTANT: Only use ONE agent per request. Pick the most appropriate agent based on the question type."
    ),
    add_handoff_back_messages=True,
    output_mode="full_history"
).compile()

print("Supervisor agent created!")
print("The supervisor can now pick the right agent for any task.")

Supervisor agent created!
The supervisor can now pick the right agent for any task.


## Section 5: Testing Agent Selection

Let's see how the supervisor identifies which agent to pick for different types of questions.

In [8]:
def test_supervisor_selection(supervisor, message):
    """Test supervisor with detailed output showing which agent is selected"""
    print(f"\nUSER QUESTION: {message}")
    print("="*60)
    print("SUPERVISOR: Analyzing question...")

    agent_selected = None
    final_response = ""

    for chunk in supervisor.stream({"messages": [{"role": "user", "content": message}]}):
        for node_name, node_update in chunk.items():
            
            # Handle intermediate agent responses
            if node_name != "__end__" and "messages" in node_update:
                if node_update["messages"]:
                    latest_message = node_update["messages"][-1]
                    if hasattr(latest_message, 'content') and latest_message.content:
                        if node_name != "supervisor":
                            print(f"SUPERVISOR SELECTED: {node_name.upper()}")
                            agent_selected = node_name
                            print(f"{node_name.upper()} RESPONSE: {latest_message.content[:100]}...")
                        else:
                            # Capture the real final supervisor answer
                            final_response = latest_message.content

        
            if node_name == "__end__":
                print("Reached end of stream")

    print(f"\nFINAL ANSWER: {final_response}")
    print(f"AGENT USED: {agent_selected.upper() if agent_selected else 'NONE'}")
    print("="*60)

    return final_response, agent_selected



# Test different types of questions to see agent selection
test_questions = [
    "What's the weather like in Tokyo?",
    "I need to calculate a 15% tip on a $120 restaurant bill",
    "Can you suggest some Italian recipes for dinner?",
    "Is it raining in Paris right now?",
    "How much should I tip for a $45 lunch?",
    "What are some popular Chinese dishes?"
]

for question in test_questions:
    test_supervisor_selection(supervisor, question)


USER QUESTION: What's the weather like in Tokyo?
SUPERVISOR: Analyzing question...
SUPERVISOR SELECTED: WEATHER_AGENT
WEATHER_AGENT RESPONSE: Successfully transferred back to supervisor...

FINAL ANSWER: The weather in Tokyo is currently cloudy with a temperature of 68°F.
AGENT USED: WEATHER_AGENT

USER QUESTION: I need to calculate a 15% tip on a $120 restaurant bill
SUPERVISOR: Analyzing question...
SUPERVISOR SELECTED: TIP_AGENT
TIP_AGENT RESPONSE: Successfully transferred back to supervisor...

FINAL ANSWER: The 15% tip on a $120 restaurant bill is $18.00, making the total amount $138.00. If you have any more questions, feel free to ask!
AGENT USED: TIP_AGENT

USER QUESTION: Can you suggest some Italian recipes for dinner?
SUPERVISOR: Analyzing question...
SUPERVISOR SELECTED: RECIPE_AGENT
RECIPE_AGENT RESPONSE: Successfully transferred back to supervisor...

FINAL ANSWER: Here are some popular Italian recipes you can try for dinner:

1. **Spaghetti Carbonara**: A classic Roman di